In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Make plots look nice
plt.style.use("dark_background")
sns.set_palette("husl")

# Show all columns when printing dataframes
pd.set_option("display.max_columns", None)
print("Libraries loaded")

Libraries loaded


### Load the data

In [3]:
# Load the raw data
df = pd.read_csv("/app/data/raw/churn.csv")
print(f"Shape of the dataset: {df.shape}")

Shape of the dataset: (7043, 21)


In [4]:
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Data types & Memory

In [5]:
# Checking missing values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    "missing_count": missing,
    "missing_pct": missing_pct
}).sort_values("missing_pct", ascending = False)

print(missing_df[missing_df["missing_count"] > 0])

Empty DataFrame
Columns: [missing_count, missing_pct]
Index: []


- There are no missing values across all the 21 columns, meaning no imputation or dropping of rows is required. The dataset is complete and ready for analysis without any data clearning needed for null values

In [6]:
df.duplicated().sum()
if df.duplicated().sum() == 0:
    print("No duplicates")

No duplicates


- No duplicate rows were found, confirming that each of the 7043 records represent a unique observation. This ensures our analysis won't be skewed by repeated entries.

In [7]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   str    
 1   gender            7043 non-null   str    
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   str    
 4   Dependents        7043 non-null   str    
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   str    
 7   MultipleLines     7043 non-null   str    
 8   InternetService   7043 non-null   str    
 9   OnlineSecurity    7043 non-null   str    
 10  OnlineBackup      7043 non-null   str    
 11  DeviceProtection  7043 non-null   str    
 12  TechSupport       7043 non-null   str    
 13  StreamingTV       7043 non-null   str    
 14  StreamingMovies   7043 non-null   str    
 15  Contract          7043 non-null   str    
 16  PaperlessBilling  7043 non-null   str    
 17  Paymen

- TotalCharges is stored as an object (string) dtype, but represents numeric values that should be of type float64. Type conversion is required before proceeding with analysis.

In [8]:
df.describe()

,SeniorCitizen,tenure,MonthlyCharges
count,7043.000000,7043.000000,7043.000000
mean,0.162147,32.371149,64.761692
std,0.368612,24.559481,30.090047
min,0.000000,0.000000,18.250000
25%,0.000000,9.000000,35.500000
50%,0.000000,29.000000,70.350000
75%,0.000000,55.000000,89.850000
max,1.000000,72.000000,118.750000


```
- Senior Citizen column should be treated as a categorical variable not numeric. This column contains binary values. The mean of 0.16 indicates that roughly 16% of customers are senior citizens. The median being 0 just confirms that the majority are not seniors. This is not a skew issue, it is he nature of binary data.

- Std of tenure is 24.56 and MonthlyCharges 30.09. MonthlyChatges have more variation than tenure meaning tenure are slighly more consistent across observations.The mean and median difference for tenure is (32.37 - 29.00) = 3.37 , Mean > Median, most customers have 29 months tenure, but long term customers push the average up to have a right skew distribution. and for MonthlyCharges (64.76-70.35) = -5.59 meaning Mean < Median, most customers pay 70+ but a small group paying a very little drags the average down to have a left skew distribution.

- The IQR for Senior Citize is 0 confirming it is a binary variable with no spread in the middle 50%, tenure is 46 and Monthly Charges is 54.35  meaning even the middle 50% show considerable variation in performance. This suggests tenure and monthly charges are not consisten across the obversations

```

In [9]:
df.columns

Index(['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents',
       'tenure', 'PhoneService', 'MultipleLines', 'InternetService',
       'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
       'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling',
       'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'],
      dtype='str')

In [10]:
categorical_features = [features for features in df.columns
                        if df[features].dtype =="str"]
print(f"Categorical Features: {categorical_features}\n")

numerical_features = [features for features in df.columns
                      if df[features].dtype !="str"]
print(f"Numerical features: {numerical_features}\n")

Categorical Features: ['customerID', 'gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges', 'Churn']

Numerical features: ['SeniorCitizen', 'tenure', 'MonthlyCharges']



In [11]:
for feature in categorical_features:
    print(f"Categories in {feature} variable: {df[feature].unique()}")
    print(f"Number of unique categories in {feature} variable: {len(df[feature].unique())}\n")

Categories in customerID variable: <StringArray>
['7590-VHVEG', '5575-GNVDE', '3668-QPYBK', '7795-CFOCW', '9237-HQITU',
 '9305-CDSKC', '1452-KIOVK', '6713-OKOMC', '7892-POOKP', '6388-TABGU',
 ...
 '9767-FFLEM', '0639-TSIQW', '8456-QDAVC', '7750-EYXWZ', '2569-WGERO',
 '6840-RESVB', '2234-XADUH', '4801-JZAZL', '8361-LTMKD', '3186-AJIEK']
Length: 7043, dtype: str
Number of unique categories in customerID variable: 7043

Categories in gender variable: <StringArray>
['Female', 'Male']
Length: 2, dtype: str
Number of unique categories in gender variable: 2

Categories in Partner variable: <StringArray>
['Yes', 'No']
Length: 2, dtype: str
Number of unique categories in Partner variable: 2

Categories in Dependents variable: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Number of unique categories in Dependents variable: 2

Categories in PhoneService variable: <StringArray>
['No', 'Yes']
Length: 2, dtype: str
Number of unique categories in PhoneService variable: 2

Categories in MultipleLi

```
customerID
- cusomerID is a unique identifier with 7043 unique values, matching the total number of rows. This column carries no predictive values and should be dropped before modeling.


Binary Columns(gender,Partner, Dependents,PhoneService, PaperlessBilling, Churn)
- These columns contain 2 unique categories(Yes/No or Male/Female) and are binary in nature. They can be encoded using label encoding or binary mapping during feature engineering.

MultipleLines
- Contains 3 categories including "No phone service", which is a subset of customers who have no phone service. This value is redundant with the "PhoneService" column and can be simplified to "No" during feature engineering.


OnlineSecurity, OnlineBackup, DeviceProtection, TechSupport, StreamingTV & StreamingMovie
- All six columns share the same 3 categories. Yes, No, No Internet service. The No internet value is redundant with the Internet service column and can be standardized to "no" during feature engineering.


Contract, Payment & InternetService
- These are nominal categorical columns with 3 and 4 categories respectively. They will require one-hot encoding during feature engineering as there is no natural order between categories.


TotalCharges
- Despite being listed as a categorical column, TotalCharges contains 6531 unique numeric values stored as strings. This confirms the earlier observation that type conversion to float64 is required before analysis.
```

### Target Variable distribution

In [12]:
# counts the raw number of each category
df["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [13]:
# converts to percentages
df["Churn"].value_counts(normalize = True) * 100

Churn
No     73.463013
Yes    26.536987
Name: proportion, dtype: float64

```

Is there class imbalance? 

Split         Severity 
90%/ 10%      Severe imbalance
80%/ 20%      Moderate imbalance
73%/ 27%      Mild-moderate imbalance
60%/ 40%      Mostly balanced

```

```
- The target variable 'Churn' is imbalanced with 73.46% of customers not churning and 26.54% churning. This is a mild-moderae class imbalance. During modeling, techniques such as SMOTE, class_weight="balanced" or "stratified sampling" maybe required to ensure the model does not become biased towards the majority class

```

In [16]:
# Categorical features Vs Target 
for feature in categorical_features:
    if feature != 'customerID' and feature != "Churn":
        print(df.groupby(feature)["Churn"].value_counts(normalize = True).unstack())
        print()

Churn         No       Yes
gender                    
Female  0.730791  0.269209
Male    0.738397  0.261603

Churn          No       Yes
Partner                    
No       0.670420  0.329580
Yes      0.803351  0.196649

Churn             No       Yes
Dependents                    
No          0.687209  0.312791
Yes         0.845498  0.154502

Churn               No       Yes
PhoneService                    
No            0.750733  0.249267
Yes           0.732904  0.267096

Churn                   No       Yes
MultipleLines                       
No                0.749558  0.250442
No phone service  0.750733  0.249267
Yes               0.713901  0.286099

Churn                  No       Yes
InternetService                    
DSL              0.810409  0.189591
Fiber optic      0.581072  0.418928
No               0.925950  0.074050

Churn                      No       Yes
OnlineSecurity                         
No                   0.582333  0.417667
No internet service  0.925950  0.

In [17]:
import_cat_features = [ "InternetService", "OnlineSecurity" "PaymentMethod", "Contract", "TechSupport"]



``` 
Contract - Strong relationship with churn. Month - to - month customers show the highest churn rate(~43%) compared to one-year (~12%) and two-year(~3%) contracts. Customers on longer contracts are significantly less likely to churn.

InternetService - Fiber optic customers show a noticeably higher churn rate compared to DSL and no internet customers sugegsting possible dissatisfaction with the service or pricing.

Gender, PhoneService - No signifanct difference in churn rate across categories. These features are unlikely to be strong predictors of churn.

```

In [20]:
# Numerical features Vs Target 
for feature in numerical_features:
    print(df.groupby("Churn")[feature].mean())
    print()

Churn
No     0.128721
Yes    0.254682
Name: SeniorCitizen, dtype: float64

Churn
No     37.569965
Yes    17.979133
Name: tenure, dtype: float64

Churn
No     61.265124
Yes    74.441332
Name: MonthlyCharges, dtype: float64



```
Senior Citizen 
Non-churned customers --> 12.87% are senior citizens 
Churned customers --> 25.47% are senior citizens

**Among non-churned customers only 12.87% are senior citizens. However, among churned customers 25.47% are senior citizens, nearly double the rate. This suggests that SeniorCitizen in a meaningful predictor of churn and should be retained as a feature during modeling.**


tenure
Non- churned customers stayed on average 37.5 months
churned customers stayed on average 17.8 months

**Among customers who leave do so ealry, the longer someone stays the less likely they are to churn.**

MonthlyCharges
Non-churned customers pay on average $61.3
churned customer pay on average $74.44

**Among customers who pay more per month are more likely to churn**

```